# Reading MCAP in a DataFrame with Daft

[MCAP](https://mcap.dev/) is the Foxglove-backed open standard for multimodal robotics logs — synchronized IMU, LiDAR, camera, ROS, and other message streams in a single file, schema-aware and replayable. Daft reads MCAP natively: every message becomes a row in a typed DataFrame, topic filters and time windows push down into the scanner, and a `topic_start_time_resolver` callback lets you align per-topic start times to keyframes.

This notebook walks the API against a real robotics recording — a 1.19 GB MCAP from the [DapengFeng/MCAP](https://huggingface.co/datasets/DapengFeng/MCAP) dataset on HuggingFace (CC-BY-NC-4.0). The FAST-LIVO `hku1` recording has four topics (Livox LiDAR, Livox IMU, stereo compressed cameras from HKU's campus) and 29,702 messages over 127.7 seconds.

> The MCAP file is ~1.2 GB and will be cached under `~/.cache/huggingface/`. Set `HF_HOME` before running if you want it elsewhere. Direct HTTP reads (`daft.read_mcap("https://...")`) are tracked in [Daft #6983](https://github.com/Eventual-Inc/Daft/issues/6983) — until that lands, download via `huggingface_hub` first.

In [ ]:
%pip install -q 'daft[mcap]>=0.7.13' 'huggingface_hub>=0.23' ipywidgets

In [8]:
from huggingface_hub import hf_hub_download

mcap_path = hf_hub_download(
    repo_id="DapengFeng/MCAP",
    repo_type="dataset",
    filename="FAST-LIVO/hku1/hku1_0.mcap",
)
print(mcap_path)

/Users/everettkleven/.cache/huggingface/hub/datasets--DapengFeng--MCAP/snapshots/579b47b2fef6b478dbd59947826ed27761bcdb60/FAST-LIVO/hku1/hku1_0.mcap


## 1. Open as a DataFrame

`daft.read_mcap` returns a lazy DataFrame. The schema is fixed across every MCAP file:

| column | type | meaning |
|--------|------|---------|
| `topic` | Utf8 | MCAP channel topic name (e.g. `/livox/imu`) |
| `log_time` | Int64 | message log time in nanoseconds since epoch |
| `publish_time` | Int64 | message publish time in nanoseconds since epoch |
| `sequence` | Int32 | per-channel sequence number |
| `data` | Utf8 | raw message bytes as a Python-style byte string |

Note: `data` keeps the row self-contained but is not the place to decode camera frames or point clouds — narrow with `topics=[...]` and run a UDF after.

In [9]:
import daft

df = daft.read_mcap(mcap_path)
df.schema()

column_name,type
topic,String
log_time,Int64
publish_time,Int64
sequence,Int32
data,String


## 2. What's in this recording?

Group by topic to count messages and bound the time range per channel. We project away `data` first so we never materialize the camera payloads.

In [10]:
(
    df.select("topic", "sequence", "log_time")
    .groupby("topic")
    .agg(
        daft.col("sequence").count().alias("messages"),
        daft.col("log_time").min().alias("t_start_ns"),
        daft.col("log_time").max().alias("t_end_ns"),
    )
    .sort("messages", desc=True)
    .show()
)

topicString,messagesUInt64,t_start_nsInt64,t_end_nsInt64
/livox/imu,25871,1577836978701916933,1577837106395623922
/left_camera/image/compressed,1277,1577836978699826955,1577837106299479007
/right_camera/image/compressed,1277,1577836978699826955,1577837106299479007
/livox/lidar,1277,1577836978699826956,1577837106299479008


Expected: `/livox/imu` ≈ 25,871 messages, the three other topics ≈ 1,277 each.

## 3. Topic filter pushes down into the scanner

Passing `topics=[...]` to `read_mcap` means the MCAP reader walks only those channels. No camera payloads get pulled into memory.

In [11]:
imu_df = daft.read_mcap(mcap_path, topics=["/livox/imu"])
print("IMU messages:", imu_df.count_rows())

[00:00] 🗡️ 🐟 Read MCAPSource (Python): 25,871 rows out, 0 B read | 🗡️ 🐟 Count: 25,871 rows in, 1 rows out | 🗡️ 🐟 Rename & Reorder: 1 rows in, 1 rows out

IMU messages: 25871


## 4. Time-window a stream

`start_time` and `end_time` are in nanoseconds, same unit as `log_time`. Anchor a 10-second window to the IMU's first log time.

In [12]:
NS_PER_S = 1_000_000_000

t0 = imu_df.agg(daft.col("log_time").min().alias("t0")).to_pydict()["t0"][0]
print("IMU t0 (ns):", t0)

window = daft.read_mcap(
    mcap_path,
    topics=["/livox/imu"],
    start_time=t0,
    end_time=t0 + 10 * NS_PER_S,
)
print("IMU messages in first 10s:", window.count_rows())

[00:00] 🗡️ 🐟 Read MCAPSource (Python): 25,871 rows out, 0 B read | 🗡️ 🐟 Min: 25,871 rows in, 1 rows out

IMU t0 (ns): 1577836978701916933
[00:00] 🗡️ 🐟 Read MCAPSource (Python): 2,028 rows out, 0 B read | 🗡️ 🐟 Count: 2,028 rows in, 1 rows out | 🗡️ 🐟 Rename & Reorder: 1 rows in, 1 rows out

IMU messages in first 10s: 2028


## 5. Per-topic keyframe alignment

`topic_start_time_resolver` is a callable that takes a file path and returns `{topic: start_time_ns}`. The reader expands one scan task per `(file, topic)`, each starting at its own offset.

For video topics, that's the natural place to plug in your keyframe table — start decoding from an I-frame, not from the file head. Here we just skip the first 5 seconds of each topic relative to the file start to show the mechanism.

In [13]:
def keyframe_resolver(file_path):
    """Return per-topic start offsets, in nanoseconds since epoch.

    In production this is where you'd look up an I-frame index next to each
    recording — keyframe-aligned decoding for video topics, sensor-startup
    masking for IMU/LiDAR."""
    return {
        "/livox/imu": t0 + 5 * NS_PER_S,
        "/livox/lidar": t0 + 5 * NS_PER_S,
    }


aligned = daft.read_mcap(
    mcap_path,
    topics=["/livox/imu", "/livox/lidar"],
    topic_start_time_resolver=keyframe_resolver,
)

(
    aligned.select("topic", "log_time")
    .groupby("topic")
    .agg(
        daft.col("log_time").min().alias("first_log_time_ns"),
        daft.col("log_time").count().alias("messages"),
    )
    .show()
)

topicString,first_log_time_nsInt64,messagesUInt64
/livox/lidar,1577836983799504995,1226
/livox/imu,1577836983702841043,24856


## 6. Time-aligned join: IMU samples near every LiDAR sweep

The kind of query that's painful in raw MCAP — for every LiDAR frame, pull the IMU samples within ±50 ms. Bucket both streams by 50 ms, join on the bucket, project the timestamps and sequence numbers.

This is the metadata-level join. Decoding the LiDAR point cloud or fusing IMU rotation is a UDF step over the result.

In [14]:
BUCKET_NS = 50_000_000  # 50 ms

imu = (
    daft.read_mcap(mcap_path, topics=["/livox/imu"])
    .select(
        daft.col("log_time").alias("imu_ts"),
        daft.col("sequence").alias("imu_seq"),
    )
    .with_column("bucket", daft.col("imu_ts") // BUCKET_NS)
)

lidar = (
    daft.read_mcap(mcap_path, topics=["/livox/lidar"])
    .select(
        daft.col("log_time").alias("lidar_ts"),
        daft.col("sequence").alias("lidar_seq"),
    )
    .with_column("bucket", daft.col("lidar_ts") // BUCKET_NS)
)

joined = lidar.join(imu, on="bucket").select("lidar_seq", "lidar_ts", "imu_seq", "imu_ts")

print("matched (lidar, imu) pairs:", joined.count_rows())
joined.sort("lidar_seq").show(5)

lidar_seqInt32,lidar_tsInt64,imu_seqInt32,imu_tsInt64
0,1577836983699663162,0,1577836983652827978
0,1577836983699663162,0,1577836983657896042
0,1577836983699663162,0,1577836983662841082
0,1577836983699663162,0,1577836983667887211
0,1577836983699663162,0,1577836983672842026


## What you'd build next

- **Decode payloads in a UDF.** Register a per-topic decoder once (e.g. `cv2.imdecode` for compressed images, the Livox `CustomMsg` parser for LiDAR), call it on `data` only after filtering. Daft distributes the work across the cluster.
- **Write filtered windows to a lakehouse.** Push the IMU/LiDAR join into an Iceberg table for downstream labeling, or land per-trip event tables in Delta.
- **Same code, fleet scale.** Point `daft.read_mcap` at an S3 prefix instead of a single file — the non-seekable stream wrapper means S3, GCS, and Azure all behave identically.

### References

- [`daft.read_mcap` docs](https://docs.daft.ai/) — full API reference
- [mcap.dev](https://mcap.dev/) — format spec, CLI, decoder libraries
- [foxglove.dev](https://foxglove.dev/) — visualization for the recordings you build queries on top of
- [DapengFeng/MCAP](https://huggingface.co/datasets/DapengFeng/MCAP) — the SLAM dataset used in this notebook
- [Daft PR #5886](https://github.com/Eventual-Inc/Daft/pull/5886) — the `topic_start_time_resolver` and non-seekable reader landing in v0.7.2